In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
target_df=spark.table("voltgrid.silver.vehicles")

In [0]:
df=spark.table("voltgrid.bronze.vehicles")

In [0]:
history_records_df=target_df.filter(F.col("active_flg")=="N")

In [0]:
source_df=df.withColumn("rnk",F.row_number().over(Window.partitionBy("vehicle_id").orderBy(F.desc("ingestion_time")))).filter(F.col('rnk')==1).withColumn('make',F.upper(F.trim('make'))).withColumn('model',F.upper(F.trim('model'))).drop("rnk","ingestion_time","source_path")

In [0]:
hash_keys=['make','model','year','battery_capacity_kwh','range_km','vehicle_type','registration_state','partner_id','effective_from','effective_to']
source_df=source_df.withColumn('hash_key',F.xxhash64(*[F.trim(F.col(c)) for c in hash_keys]))
target_df=target_df.withColumn('hash_key',F.xxhash64(*[F.trim(F.col(c)) for c in hash_keys]))

In [0]:
target_df=target_df.filter(F.col("active_flg")=="Y")

In [0]:
changed_records=target_df.alias("tr").join(source_df.alias("sr"),F.col("tr.vehicle_id")==F.col("sr.vehicle_id"),"inner").filter(F.col("tr.hash_key") != F.col("sr.hash_key")).select(F.col("tr.*")).withColumn('active_flg',F.lit('N')).withColumn('end_date',F.current_date()).drop("hash_key")

In [0]:
source_df=source_df.drop("hash_key")
target_df=target_df.drop("hash_key")

In [0]:
old_records=target_df.alias("tr").join(changed_records.alias("sr"),F.col("tr.vehicle_id")==F.col("sr.vehicle_id"),"left_anti")

In [0]:
new_records=source_df.alias("sr").join(target_df.alias("tr"),F.col("tr.vehicle_id")==F.col("sr.vehicle_id"),"left_anti").withColumn('active_flg',F.lit('Y')).withColumn('start_date',F.current_date()).withColumn('end_date',F.lit(None).cast('date'))

In [0]:
changed_records_val=source_df.alias('sr').join(changed_records.alias('tr'),F.col('sr.vehicle_id')==F.col('tr.vehicle_id'),'left_semi').withColumn('active_flg',F.lit('Y')).withColumn('start_date',F.current_date()).withColumn('end_date',F.lit(None).cast('date'))

In [0]:
total_records = (
    history_records_df.unionByName(changed_records)
    .unionByName(changed_records_val)
    .unionByName(new_records)
    .unionByName(old_records)
)

In [0]:
total_records.write.format('delta').mode('overwrite').option('overwriteSchema','true').saveAsTable('voltgrid.silver.vehicles')